# Tutorial: Estimating Social Contact Matrices with BRCfine

In [ ]:
import numpy as np
import pandas as pd

from jax.random import PRNGKey
from numpyro.infer.autoguide import AutoNormal

from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import ParticipantGenerator, MatrixGenerator, ContactGenerator, Subgroup
from cntmosaic.vis import plot_mosaic, plot_mosaic_marginal

from cntmosaic.dataloader import CoordToColumns, DataLoader

from cntmosaic.models import BRCfine
from cntmosaic.models.priors import Spline2D, PSpline2D

from cntmosaic.analysis import ModelSummariserBRC, ModelEvaluatorBRC

import altair as alt

## 2. Synthetic Data Generation
We'll use `cntmosaic`'s data generation tools to create realistic synthetic contact data. For details, refer to the "Tutorial: Generating Synthetic Contact Data".

We first load the contact pattern templates and population data:

In [ ]:
templates = load_template_patterns(country='United_States')
df_age_dist = load_age_distribution(country='United_States')

age_dist = df_age_dist.P.values

We define a subgroup with 1000 participants with the `Subgroup` class:

In [ ]:
my_population = Subgroup(
  n=1000,                       # Generate 1000 participants
  age_dist=age_dist,            # Use this age distribution
  mean_cint_margin=15.0,        # Average 15 contacts per person
  label='general'
)

We randomly simulate a contact intensity matrix using the `MatrixGenerator` class.

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
contact_matrix = matrix_gen.generate_single(
  subgroup=my_population,
  seed=42   # For reproducibility
)

print("Matrix shape:", contact_matrix.shape)
print("Average contacts per person:", contact_matrix.sum(axis=0).mean())

Visualize the generated contact matrix with `plot_mosaic` function:

In [ ]:
plot_mosaic(contact_matrix, title='Synthetic Contact Intensity Matrix')

The `ParticipantGenerator` class is then used to simulate participant data based on the defined subgroup.

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(my_population)

# Generate participants
df_part = part_gen.generate(seed=42)

df_part.head()

The `ContactGenerator` class simulates contact data for the participants:

In [ ]:
cnt_gen = ContactGenerator(
  df_part=df_part,
  cint_matrices=contact_matrix,
  model='poisson'
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
df_cnt.head()

Plot the empirical contact matrix from the simulated contact data:

In [ ]:
emp_cint_matrix = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)
plot_mosaic(emp_cint_matrix, title='Empirical Contact Matrix', zlabel='Intensity')

In [ ]:
col_map = CoordToColumns(
  age_part="age_group",
  age_cnt="age_cnt",
  age_pop="age",
  size_pop="P"
)

dataloader = DataLoader(
  df_part,
  df_cnt,
  df_age_dist,
  col_map=col_map
)

In [ ]:
priors = {
  "rate": PSpline2D(prior_type="global", order=2, grid_type="diff-age", tau_ratio=2.0)
}

brc = BRCfine(
  dataloader=dataloader,
  priors=priors
)

In [ ]:
guide = AutoNormal(brc.model)
brc.run_inference_svi(prng_key=PRNGKey(0), guide=guide, num_steps=20_000)

In [ ]:
summariser = ModelSummariserBRC(brc)

In [ ]:
summary_cint = summariser.summarise_cint()

cint_median = summary_cint[1]
plot_mosaic(cint_median, title='Inferred Contact Intensity Matrix', zlabel='Intensity')

In [ ]:
mcint_q = summariser.summarise_mcint()
mcint_median = mcint_q[1]
mcint_lower = mcint_q[0]
mcint_upper = mcint_q[2]

plot_mosaic_marginal(
  mcint_median,
  mcint_lower,
  mcint_upper,
  title='Inferred Marginal Contact Intensity'
)

### 3.7 Evaluating Model Accuracy
The accuracy of the estimated contact matrices can be evaluated by comparing them to the true contact matrices used in the synthetic data generation. The `ModelEvaluatorSocialMix` class can be used to compute a wide range of accuracy metrics.

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluatorBRC(summariser, cint_matrix_true=contact_matrix)

In [ ]:
# Default metrics
evaluator.evaluate()